In [ ]:
# Run once if not in Colab
!pip install huggingface_hub
!pip install transformers datasets tokenizers seqeval -q evaluate


In [1]:
from utilities_2 import preprocess, create_model_init, get_compute_metrics
from huggingface_hub import HfApi
import os
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)
import numpy as np

from seqeval.metrics import f1_score
from transformers import EvalPrediction

from transformers import TrainingArguments
from datasets import get_dataset_config_names
from datasets import load_dataset

from transformers import DataCollatorForTokenClassification

from collections import defaultdict
from datasets import DatasetDict

from collections import Counter

from transformers import Trainer

import torch.nn as nn
#from matplotlib import pyplot as plt
from transformers import XLMRobertaConfig
from transformers.modeling_outputs import TokenClassifierOutput
from transformers.models.roberta.modeling_roberta import RobertaModel
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel

import torch

from seqeval.metrics import classification_report

In [2]:
#You can mis this
print(torch.cuda.is_available())  # Should return True if CUDA is properly set up.
print(torch.cuda.get_device_name(0))  # Should return the GPU name if detected.

True
NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
# And this
# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create a tensor and move it to GPU
#x = torch.rand(5, 5).to(device)  # Move tensor to GPU
#print(x)

In [2]:
# get data
model_name="xlm-roberta-large"
language_code="ru"
#data=preprocess(language_code=language_code, model_name=model_name, train=True)
#tokenized_datasets, label_list, label2id, id2label, tokenizer= data
#print(tokenized_datasets["train"][0])

Get data

In [3]:
# --- Check for GPU availability ---
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Using CPU.")


CUDA is available. Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [4]:
try:
    data = preprocess(language_code=language_code, model_name=model_name, train=True)
    tokenized_dataset, label_list, label2id, id2label, tokenizer = data[0], data[1], data[2], data[3], data[4]
    print("Sample from training data:")
    print(tokenized_dataset["train"][0])
    num_labels = len(label_list)
except NameError:
    print("Error: The 'preprocess' function is not defined.")
    print("Please ensure 'utilities.py' is in the same directory or accessible in your Python path,")
    print("and that it contains the 'preprocess' function.")
    # Example placeholder data if preprocess fails - replace with actual loading if needed
    tokenized_dataset = None # Set to None or load dummy data
    label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"] # Example labels
    label2id = {label: i for i, label in enumerate(label_list)}
    id2label = {i: label for i, label in enumerate(label_list)}
    num_labels = len(label_list)
    tokenizer = AutoTokenizer.from_pretrained(model_name) # Load tokenizer separately if needed
    print("\nWARNING: Using placeholder data because 'preprocess' failed.")

Map:   0%|          | 0/7560 [00:00<?, ? examples/s]

Map:   0%|          | 0/5152 [00:00<?, ? examples/s]

Sample from training data:
{'input_ids': [0, 417, 174222, 34800, 27224, 4988, 183, 27145, 151609, 61, 85063, 244, 29524, 3280, 6, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [-100, 10, 1, 10, -100, 10, 10, -100, 10, 10, 10, -100, -100, -100, 10, -100, -100, -100, -100, -100, -100, -100, -1

Build model

In [ ]:
num_epochs = 3
batch_size = 
logging_steps = len(tokenized_dataset["train"]) // batch_size
new_model_name = f"{model_name}-finetuned-{language_code}"
model_init=create_model_init(model_name=model_name, data=data, device=device)
compute_metrics=get_compute_metrics(id2label=id2label)

data_collator = DataCollatorForTokenClassification(tokenizer)
training_args = TrainingArguments(
    output_dir=new_model_name, log_level="error", num_train_epochs=num_epochs, 
    per_device_train_batch_size=batch_size, 
    per_device_eval_batch_size=batch_size,
    save_steps=1e6, weight_decay=0.01, disable_tqdm=False, 
    logging_steps=logging_steps, push_to_hub=False)

trainer = Trainer(model_init=model_init, args=training_args, 
                  data_collator=data_collator, compute_metrics=compute_metrics,
                  train_dataset=tokenized_dataset["train"],
                  eval_dataset=tokenized_dataset["validation"], 
                  tokenizer=tokenizer)

C:\Users\user\AppData\Local\Temp\ipykernel_4448\1523952701.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model_init=model_init, args=training_args,


Train

In [6]:
trainer.train()

Step,Training Loss


KeyboardInterrupt: 

Save

In [ ]:
#Save locally
trainer.save_model("./my_ner_model")
tokenizer.save_pretrained("./my_ner_model")

('./my_ner_model\\tokenizer_config.json',
 './my_ner_model\\special_tokens_map.json',
 './my_ner_model\\tokenizer.json')

In [ ]:
# Only in VScode
! pip install ipywidgets

  Using cached ipywidgets-8.1.6-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.14-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.14-py3-none-any.whl.metadata (4.1 kB)
Using cached ipywidgets-8.1.6-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.14-py3-none-any.whl (213 kB)
Using cached widgetsnbextension-4.0.14-py3-none-any.whl (2.2 MB)


In [3]:
from huggingface_hub import interpreter_login

interpreter_login()


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|



In [ ]:
#If you have it locally
model = AutoModelForTokenClassification.from_pretrained("./my_ner_model")
tokenizer = AutoTokenizer.from_pretrained("./my_ner_model")

In [11]:
#Save to HF
model.push_to_hub("OOOss/nlp_exp") #"your-username/your-model-name"
tokenizer.push_to_hub("OOOss/nlp_exp") #"your-username/your-model-name"

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

c:\Users\user\anaconda3\envs\nlp_gpu\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--OOOss--nlp_exp. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/OOOss/nlp_exp/commit/9922648dd40ccca828ba05e72b29ea1af074f4b9', commit_message='Upload tokenizer', commit_description='', oid='9922648dd40ccca828ba05e72b29ea1af074f4b9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/OOOss/nlp_exp', endpoint='https://huggingface.co', repo_type='model', repo_id='OOOss/nlp_exp'), pr_revision=None, pr_num=None)

Only junk below

Create model

In [ ]:
from transformers import AutoTokenizer,AutoModelForTokenClassification,AutoModelForTokenClassification, AutoConfig
from transformers import TrainingArguments, Trainer, IntervalStrategy
from transformers import DataCollatorForTokenClassification
from transformers import pipeline
from huggingface_hub import HfApi


config = AutoConfig.from_pretrained(model_name, num_labels=len(label_list) , id2label=id2label, label2id=label2id)
model = AutoModelForTokenClassification.from_config(config)
data_collator = DataCollatorForTokenClassification(tokenizer)

args = TrainingArguments(
"test-ner",
#evaluation_strategy = "epoch",
learning_rate=2e-5,
per_device_train_batch_size=16,
per_device_eval_batch_size=16,
num_train_epochs=1,
weight_decay=0.01
)


In [16]:
from utilities import preprocess
# get data
model_name="xlm-roberta-large"
data=preprocess(language_code='ru', model_name=model_name)
tokenized_datasets_ru, label_list, label2id, id2label, tokenizer= data[0], data[1],data[2], data[3], data[4]
print(tokenized_datasets_ru["train"][0])


Map:   0%|          | 0/7560 [00:00<?, ? examples/s]

Map:   0%|          | 0/5152 [00:00<?, ? examples/s]

Map:   0%|          | 0/10692 [00:00<?, ? examples/s]

{'tokens': ['В', 'Пакистан', 'протестовать', 'против', 'отмена', 'приговор', 'за', 'богохульство', '.'], 'ner_tags': [10, 1, 10, 10, 10, 10, 10, 10, 10], 'input_ids': [0, 417, 174222, 34800, 27224, 4988, 183, 27145, 151609, 61, 85063, 244, 29524, 3280, 6, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 

In [24]:

# intitialize model
from transformers import AutoTokenizer,AutoModelForTokenClassification,AutoModelForTokenClassification, AutoConfig
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
from transformers import pipeline

config = AutoConfig.from_pretrained(model_name, num_labels=len(label_list) , id2label=id2label, label2id=label2id)
model = AutoModelForTokenClassification.from_config(config)
data_collator = DataCollatorForTokenClassification(tokenizer) 
args = TrainingArguments(
"test-ner",
evaluation_strategy = "epoch",
learning_rate=2e-5,
per_device_train_batch_size=16,
per_device_eval_batch_size=16,
num_train_epochs=3,
weight_decay=0.01,
)


C:\Users\maryz\AppData\Roaming\Python\Python311\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# train

trainer = Trainer(
    model,
    args,
   train_dataset=tokenized_datasets_ru["train"],
   eval_dataset=tokenized_datasets_ru["validation"],
   data_collator=data_collator,
   tokenizer=tokenizer
)
trainer.train()

In [1]:
import torch # Import torch library
from utilities import preprocess
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)

# --- Configuration ---
model_name = "xlm-roberta-large"
language_code = 'ru'
output_dir = "test-ner" # Define output directory for TrainingArguments
learning_rate = 2e-5
train_batch_size = 4
eval_batch_size = 4
num_train_epochs = 1
weight_decay = 0.01

# --- Check for GPU availability ---
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Using CPU.")

# --- Data Preprocessing ---
# Assuming 'utilities.preprocess' returns the necessary components
# Make sure the preprocess function is defined or imported correctly
try:
    data = preprocess(language_code=language_code, model_name=model_name, train=True)
    tokenized_datasets_ru, label_list, label2id, id2label, tokenizer = data[0], data[1], data[2], data[3], data[4]
    print("Sample from training data:")
    print(tokenized_datasets_ru["train"][0])
    num_labels = len(label_list)
except NameError:
    print("Error: The 'preprocess' function is not defined.")
    print("Please ensure 'utilities.py' is in the same directory or accessible in your Python path,")
    print("and that it contains the 'preprocess' function.")
    # Example placeholder data if preprocess fails - replace with actual loading if needed
    tokenized_datasets_ru = None # Set to None or load dummy data
    label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"] # Example labels
    label2id = {label: i for i, label in enumerate(label_list)}
    id2label = {i: label for i, label in enumerate(label_list)}
    num_labels = len(label_list)
    tokenizer = AutoTokenizer.from_pretrained(model_name) # Load tokenizer separately if needed
    print("\nWARNING: Using placeholder data because 'preprocess' failed.")

print(f"Type of tokenized_datasets_ru: {type(tokenized_datasets_ru)}")
print(f"Keys in tokenized_datasets_ru: {tokenized_datasets_ru.keys()}")
print(f"Sample item from train: {tokenized_datasets_ru['train'][0]}")

# --- Initialize Model ---
print(f"\nInitializing model: {model_name}")
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
# Load the model
model = AutoModelForTokenClassification.from_pretrained(model_name, config=config)

# --- Move Model to GPU (if available) ---
# Explicitly move the model to the selected device (GPU or CPU)
model.to(device)
print(f"Model moved to device: {device}")

# --- Data Collator ---
# Handles padding and batch preparation.
# The Trainer will automatically move data batches to the correct device.
data_collator = DataCollatorForTokenClassification(tokenizer)

# --- Training Arguments ---
# Configure training parameters.
# `TrainingArguments` automatically detects and uses CUDA if available by default.
# Setting `no_cuda=False` (default) ensures GPU is used if possible.
args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    weight_decay=0.01,
    fp16=True,  # mixed precision
    logging_dir='./logs',
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True
)

# --- Initialize Trainer ---
# The Trainer class handles the training loop, evaluation, and device placement.
if tokenized_datasets_ru: # Check if data was loaded successfully
    trainer = Trainer(
        model=model, # The model is already on the correct device
        args=args,
        train_dataset=tokenized_datasets_ru["train"],
        eval_dataset=tokenized_datasets_ru["validation"],
        data_collator=data_collator,
        tokenizer=tokenizer
    )

    # --- Train the Model ---
    print("\nStarting training...")
    trainer.train()
    print("Training finished.")

    # --- Save the final model ---
    trainer.save_model(f"{output_dir}/final_model")
    tokenizer.save_pretrained(f"{output_dir}/final_model")
    print(f"Model saved to {output_dir}/final_model")

    # --- Example of using the trained model (optional) ---
    print("\nExample prediction using the trained model:")
    # Load the fine-tuned model and tokenizer
    # The pipeline automatically handles moving data to the model's device
    ner_pipeline = pipeline(
        "token-classification",
        model=f"{output_dir}/final_model",
        tokenizer=f"{output_dir}/final_model",
        # device=0 if torch.cuda.is_available() else -1 # Explicitly set device for pipeline if needed
    )

    example_text = "Пример текста на русском языке для проверки NER." # Example text in Russian
    results = ner_pipeline(example_text)
    print(f"Input: {example_text}")
    print(f"Predictions: {results}")

else:
    print("\nSkipping training because data preprocessing failed.")



c:\Users\user\anaconda3\envs\nlp_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA is available. Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU


Map: 100%|██████████| 5152/5152 [00:00<00:00, 7455.55 examples/s]


Sample from training data:
{'tokens': ['В', 'Пакистан', 'протестовать', 'против', 'отмена', 'приговор', 'за', 'богохульство', '.'], 'ner_tags': [10, 1, 10, 10, 10, 10, 10, 10, 10], 'input_ids': [0, 417, 174222, 34800, 27224, 4988, 183, 27145, 151609, 61, 85063, 244, 29524, 3280, 6, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

C:\Users\user\AppData\Local\Temp\ipykernel_12292\1870445421.py:101: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
0,0.068700,0.263126


Training finished.
Model saved to test-ner/final_model

Example prediction using the trained model:


Device set to use cuda:0


Input: Пример текста на русском языке для проверки NER.
Predictions: [{'entity': 'B-ORG', 'score': np.float32(0.612514), 'index': 9, 'word': 'NER', 'start': 44, 'end': 47}]


In [2]:
args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="steps",
    eval_steps=500,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    logging_dir='./logs',
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True
)


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:


# save prediction
predictions=trainer.predict(tokenized_datasets_ru["test"])
final_predictions_test = [
    [id2label[p] for p in sentence] for sentence in predictions
]

iob_predictions=[]
for sent_idx, final_pred in enumerate(final_predictions_test):

    tokens = tokenized_datasets_ru["test"]["tokens"][sent_idx]  # Tokens from your dataset
    tokens_ids = tokenized_datasets_ru["test"]["input_ids"][sent_idx]  # Tokens from your dataset
    sentence_iob = []
    pred_label_cleaned=final_pred[1:len(tokens)+1]
    for token_idx, token in enumerate(tokens):
        pred_label = pred_label_cleaned[token_idx]  # Get the label from final_predictions
        sentence_iob.append(f"{token_idx+1}\t{tokens[token_idx]}\t{pred_label}")

    iob_predictions.append(sentence_iob)

with open("ner_predictions.iob", "w") as f:
    for sentence in iob_predictions:
        for line in sentence:
            f.write(f"{line}\n")
        f.write("\n")  # Separate sentences with a blank line
#save models

from huggingface_hub import HfApi

trainer.save_model("./my_ner_model")
tokenizer.save_pretrained("./my_ner_model")
# Load your model and tokenizer
model = AutoModelForTokenClassification.from_pretrained("./my_ner_model")
tokenizer = AutoTokenizer.from_pretrained("./my_ner_model")

# Push to the hub
model.push_to_hub("your-username/your-model-name")
tokenizer.push_to_hub("your-username/your-model-name")


In [ ]:
data = preprocess(language_code=language_code, model_name=model_name, train=False)
tokenized_datasets["test"]=data[0]
predictions=trainer.predict(tokenized_datasets["test"])
predictions = predictions.predictions.argmax(2)

In [ ]:
converted_predictions_labels = [
    [id2label[p] for p in sentence] for sentence in predictions
  ]
iob_predictions=[]
for sent_idx, final_pred in enumerate(converted_predictions_labels):

    tokens = tokenized_datasets["test"]["tokens"][sent_idx]  # Tokens from your dataset
    #tokens_ids = tokenized_datasets["test"]["input_ids"][sent_idx]  # Tokens from your dataset # This line is not needed
    sentence_iob = []
    #pred_label_cleaned=final_pred[1:len(tokens)+1] # Incorrect slicing, causing the error
    pred_label_cleaned = final_pred[1:len(final_pred) - 1] # Adjusting slicing to align with tokens

    # Ensure both lists have the same length for proper iteration
    min_len = min(len(tokens), len(pred_label_cleaned))
    for token_idx in range(min_len):
        pred_label = pred_label_cleaned[token_idx]  # Get the label from final_predictions
        sentence_iob.append(f"{token_idx+1}\t{tokens[token_idx]}\t{pred_label}")

    iob_predictions.append(sentence_iob)


In [ ]:
with open("ner_predictions.iob", "w") as f:
    for sentence in iob_predictions:
        for line in sentence:
            f.write(f"{line}\n")
        f.write("\n")  # Separate sentences with a blank line

In [1]:
def save_preds(trainer,language_code:  str, target_language: str, tokenized_dataset, id2label):
  predictions=trainer.predict(tokenized_dataset["test"])
  predictions = predictions.predictions.argmax(2)

  converted_predictions_labels = [
      [id2label[p] for p in sentence] for sentence in predictions
  ]
  iob_predictions=[]
  for sent_idx, final_pred in enumerate(converted_predictions_labels):

      tokens = tokenized_dataset["test"]["tokens"][sent_idx]  # Tokens from your dataset
      sentence_iob = []
      pred_label_cleaned = final_pred[1:len(final_pred) - 1] # Adjusting slicing to align with tokens

      # Ensure both lists have the same length for proper iteration
      min_len = min(len(tokens), len(pred_label_cleaned))
      for token_idx in range(min_len):
          pred_label = pred_label_cleaned[token_idx]  # Get the label from final_predictions
          sentence_iob.append(f"{token_idx+1}\t{tokens[token_idx]}\t{pred_label}")

      iob_predictions.append(sentence_iob)


  with open(f"preds/{language_code}_{target_language}_predictions.iob", "w") as f:
    for sentence in iob_predictions:
        for line in sentence:
            f.write(f"{line}\n")
        f.write("\n")  # Separate sentences with a blank line

In [ ]:
#from utilities import save_preds

os.makedirs("preds", exist_ok=True)

lang_list=["bg", "ru", "sl", "sl_cyrilic", "uk"]
for target_language in lang_list:
  target_lang_data=preprocess(language_code=language_code, model_name=model_name, train=False)
  pred_data, id2label = target_lang_data[0], target_lang_data[3]
  save_preds(trainer, language_code, target_language, tokenized_dataset=pred_data, id2label=id2label )
  print(target_language + " done")

Map: 100%|██████████| 10692/10692 [00:02<00:00, 3760.10 examples/s]


KeyboardInterrupt: 